# CyberForecaster — windows → baseline walkthrough

Interactive version of `src/preprocessing/pipeline.py` + `baseline_logreg`, for learning and debugging.
**Run order:** download data first (`python scripts/download_data.py --yes`), then run top-to-bottom.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

RAW = Path('..') / 'data' / 'raw'
OUT = Path('..') / 'data' / 'processed'
print('raw files:', sorted(p.name for p in RAW.glob('*.csv')))

## 1. Load + clean one day-file first (fast iteration), then all

In [ ]:
from src.ingestion.csv_loader import load_day_csv, load_many

first = sorted(RAW.glob('*.csv'))[0]
sample = load_day_csv(first)          # inspect cleaning output per file
sample.head()

**Checkpoint:** do the label counts look sane? Any `Other:` rows? If a label variant is unmapped,
extend `_canonical_label` in `csv_loader.py` — never let silent `Other:` buckets into training.

In [ ]:
flows = load_many(sorted(RAW.glob('*.csv')))
print(flows.shape)
flows['Label'].value_counts().head(20)

## 2. Windows: flows → temporal states

In [ ]:
from src.features.window_builder import build_windows, WINDOW_FEATURES

windows = build_windows(flows)
print(f'{len(windows)} windows x {len(WINDOW_FEATURES)} features')
windows[WINDOW_FEATURES + ['attack_frac']].describe().T

In [ ]:
# visual sanity: attack activity over time (this is what the demo timeline shows)
import matplotlib.pyplot as plt
ax = windows['attack_frac'].plot(figsize=(14, 3), title='attack_frac per 60s window')
plt.show()

## 3. Rule-engine validation (MITRE mapping honesty check)

In [ ]:
from src.attack_mapping.mitre_mapper import validate_rules

ct = validate_rules(windows)
# tune rule thresholds until agreement is reasonable, then FREEZE them and document.

## 4. Sequences + chronological split (leakage-safe)

In [ ]:
from src.features.window_builder import make_sequences, chrono_split

X, y_prog, y_stage, ends = make_sequences(windows)
tr, va, te = chrono_split(windows, ends)
print(f'total={len(X)} train={len(tr)} val={len(va)} test={len(te)}')
print(f'positive rate — train {y_prog[tr].mean():.3f} | val {y_prog[va].mean():.3f} | test {y_prog[te].mean():.3f}')

## 5. Logistic baseline (the PS-required benchmark)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from src.models.baseline_logreg import evaluate

Xtr, Xte = X[tr].reshape(len(tr), -1), X[te].reshape(len(te), -1)
scaler = StandardScaler().fit(Xtr)
clf = LogisticRegression(max_iter=1000, class_weight='balanced').fit(scaler.transform(Xtr), y_prog[tr].astype(int))
metrics = evaluate(y_prog[te], clf.predict_proba(scaler.transform(Xte))[:, 1])
metrics

## 6. Persist artifacts for the app + model training

In [ ]:
from src.preprocessing.pipeline import run

run(RAW, OUT)   # writes windows.parquet + sequences_{train,val,test}.npz